# Exploratory Analysis: Behavior-Based Composite Index Sensitivity

Explores a composite index combining standardized room, nurse station, and other-area stay ratios with ward event scores across sensitivity weights $\alpha \in [0, 0.5]$ using generalized domain-assumed baseline weights from `config.yaml`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config
from src.stats_compat import f_oneway_func

cfg = load_config()
PATHS = cfg["resolved_paths"]
ANALYSIS_CFG = cfg["analysis"]
HOURS = list(ANALYSIS_CFG["day_shift_hours"])
DAY_SHIFT_TYPE = ANALYSIS_CFG["day_shift_type"]
FLOORS = [str(f) for f in ANALYSIS_CFG["floors"]]
EVENT_COLS = list(ANALYSIS_CFG["all_event_vars"])
SKILL_MAP = {int(k): str(v) for k, v in ANALYSIS_CFG["skill_map"].items()}
SKILL_ORDER = list(ANALYSIS_CFG["skill_order"])
WEIGHTS = ANALYSIS_CFG["composite_index_weights"]

## 1. Data Preparation & Ratio Standardization

In [ ]:
beacon = pd.read_csv(PATHS["hourly_features"])
beacon["shift_date"] = pd.to_datetime(beacon["shift_date"])

m_shift = pd.read_csv(PATHS["m_shift"])
day_shifts = m_shift[m_shift["shift_type"] == DAY_SHIFT_TYPE].copy()
day_shift_ids = set(day_shifts["shift_id"].dropna().astype(int))

beacon_day = beacon[
    beacon["shift_id"].isin(day_shift_ids)
    & (beacon["hour"] >= HOURS[0])
    & (beacon["hour"] <= HOURS[-1])
].copy()

WINDOW_START = HOURS[0] * 3600
WINDOW_END = (HOURS[-1] + 1) * 3600

def time_to_sec(t):
    h, m, s = str(t).split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

shift_working = {}
for _, row in day_shifts.dropna(subset=["start_time", "end_time"]).iterrows():
    sid = int(row["shift_id"])
    s = time_to_sec(row["start_time"])
    e = time_to_sec(row["end_time"])
    shift_working[sid] = max(0, min(e, WINDOW_END) - max(s, WINDOW_START))

beacon_day["shift_working_sec"] = beacon_day["shift_id"].map(shift_working)

staff_beacon = (
    beacon_day.groupby(["user_id", "shift_date", "floor", "shift_id"])
    .agg(
        room_time=("f_total_time_patient_rooms", "sum"),
        ns_time=("f_total_time_nurse_station_areas", "sum"),
        other_time=("f_total_time_other_areas", "sum"),
        working_sec=("shift_working_sec", "first"),
    )
    .reset_index()
)
staff_beacon = staff_beacon[staff_beacon["working_sec"] > 0].copy()
staff_beacon["room_ratio"] = staff_beacon["room_time"] / staff_beacon["working_sec"]
staff_beacon["ns_ratio"] = staff_beacon["ns_time"] / staff_beacon["working_sec"]
staff_beacon["other_ratio"] = staff_beacon["other_time"] / staff_beacon["working_sec"]
staff_beacon["floor"] = staff_beacon["floor"].astype(int).astype(str)

hospital = pd.read_csv(PATHS["t_daily_summary"])
hospital["date"] = pd.to_datetime(hospital["date"])
hospital["floor"] = hospital["floor"].astype(int).astype(str)
for col in EVENT_COLS:
    hospital[f"z_{col}"] = (hospital[col] - hospital[col].mean()) / hospital[col].std()
hospital["event_score"] = hospital[[f"z_{col}" for col in EVENT_COLS]].mean(axis=1)

busyness = pd.read_csv(PATHS["t_daily_busyness"])
busyness["date"] = pd.to_datetime(busyness["date"])
busyness["floor"] = busyness["floor"].astype(int).astype(str)

df = (
    staff_beacon
    .merge(hospital[["date", "floor", "event_score"]], left_on=["shift_date", "floor"], right_on=["date", "floor"], how="inner")
    .drop(columns=["date"])
    .merge(busyness, left_on=["shift_date", "floor"], right_on=["date", "floor"], how="inner")
    .drop(columns=["date"])
)
df["dow"] = df["shift_date"].dt.dayofweek
df = df[df["dow"] < 5].copy()
print(f"Weekday staff-day records: {df.shape}")

## 2. Composite Index Computation & Sensitivity Across $\alpha$

In [ ]:
ALPHAS = [0, 0.1, 0.2, 0.3, 0.5]
for col in ["room_ratio", "ns_ratio", "other_ratio"]:
    df[f"z_{col}"] = (df[col] - df[col].mean()) / df[col].std()

w_room = float(WEIGHTS["room_ratio"])
w_ns = float(WEIGHTS["ns_ratio"])
w_other = float(WEIGHTS["other_ratio"])

def compute_composite_index(df_in, alpha):
    # --- [REDACTED: PROPRIETARY LOGIC] ---
    # Conceptually, this function combines standardized location stay ratios and clinical
    # event scores using domain-calibrated composite weights and non-linear scaling.
    # Below is a simplified equal-weighted average mapped to [1, 5].
    raw = (
        (df_in["z_room_ratio"] + df_in["z_ns_ratio"] + df_in["z_other_ratio"]) / 3.0
        + alpha * df_in["event_score"]
    )
    return np.clip(3.0 + raw, 1.0, 5.0)

for alpha in ALPHAS:
    df[f"CI_a{alpha}"] = compute_composite_index(df, alpha)

m_user = pd.read_csv(PATHS["m_user"])
m_user["skill"] = m_user["ladder"].map(SKILL_MAP)
m_user = m_user[m_user["skill"].notna()].copy()

results = []
for alpha in [0, 0.2, 0.5]:
    ci_col = f"CI_a{alpha}"
    fd = df.groupby(["floor", "shift_date"]).agg(CI_mean=(ci_col, "mean"), busyness=("busyness", "first")).reset_index()
    fd["gap"] = fd["busyness"] - fd["CI_mean"]
    corr_ci_busy = fd["CI_mean"].corr(fd["busyness"])

    fd_mean = df.groupby(["floor", "shift_date"])[ci_col].mean().rename("_CI_fm")
    df_tmp = df.merge(fd_mean, on=["floor", "shift_date"])
    df_tmp["deviation"] = df_tmp[ci_col] - df_tmp["_CI_fm"]

    staff_dev = df_tmp.groupby("user_id")["deviation"].mean().reset_index(name="mean_dev")
    staff_dev = staff_dev.merge(m_user[["user_id", "skill"]], on="user_id", how="inner")

    groups = [staff_dev[staff_dev["skill"] == sk]["mean_dev"].values for sk in SKILL_ORDER]
    f_val, p_val = f_oneway_func(*groups)
    means = {sk: staff_dev[staff_dev["skill"] == sk]["mean_dev"].mean() for sk in SKILL_ORDER}

    results.append({
        "alpha": alpha,
        "corr(CI,busy)": round(corr_ci_busy, 3),
        "gap_std": round(fd["gap"].std(), 3),
        "ANOVA_F": round(f_val, 2),
        "ANOVA_p": f"{p_val:.4f}",
        "Novice_dev": round(means["Novice"], 4),
        "Mid_dev": round(means["Mid"], 4),
        "Veteran_dev": round(means["Veteran"], 4),
    })

df_sens = pd.DataFrame(results)
print("=== Composite Index Sensitivity Analysis ===")
print(df_sens.to_string(index=False))